In [1]:
import coacd
import trimesh
from pathlib import Path

input_file = Path("data/laso_grasp_only_affordance/mug/1ae1ba5dfb2a085247df6165146d5bbd/reconstructed_mesh.ply").expanduser().resolve()
mesh = trimesh.load(input_file, force="mesh")
mesh = coacd.Mesh(mesh.vertices, mesh.faces)
# parts = coacd.run_coacd(mesh) # a list of convex hulls.
parts = coacd.run_coacd(
    mesh,
    threshold=0.03,
    max_convex_hull=64,
    preprocess_resolution=40,
    resolution=1500,
    mcts_nodes=10,
    mcts_iterations=30,
    mcts_max_depth=3,
    seed=42,
)


# Or use real metric mode (threshold in meters)
# parts = coacd.run_coacd(mesh, threshold=0.01, real_metric=True)

[2026-10-27 04:48:17.404] [CoACD] [info] threshold               0.03
[2026-10-27 04:48:17.404] [CoACD] [info] max # convex hull       64
[2026-10-27 04:48:17.404] [CoACD] [info] preprocess mode         auto
[2026-10-27 04:48:17.404] [CoACD] [info] preprocess resolution   40
[2026-10-27 04:48:17.404] [CoACD] [info] pca                     false
[2026-10-27 04:48:17.404] [CoACD] [info] mcts max depth          3
[2026-10-27 04:48:17.404] [CoACD] [info] mcts nodes              10
[2026-10-27 04:48:17.404] [CoACD] [info] mcts iterations         30
[2026-10-27 04:48:17.404] [CoACD] [info] merge                   true
[2026-10-27 04:48:17.404] [CoACD] [info] decimate                false
[2026-10-27 04:48:17.404] [CoACD] [info] max_ch_vertex           256
[2026-10-27 04:48:17.404] [CoACD] [info] extrude                 false
[2026-10-27 04:48:17.404] [CoACD] [info] extrude margin          0.01
[2026-10-27 04:48:17.404] [CoACD] [info] approximate mode        ch
[2026-10-27 04:48:17.404] [CoAC

In [ ]:
# save as multiple collision meshes
import numpy as np
# object_id = input_file.parent.name
# object_id = input_file.parent / "collision_meshes"
# OUTDIR = 'outputs/coacd_parts'
# print(object_id)
# for index, (vertices, faces) in enumerate(parts):
#     part_mesh = trimesh.Trimesh(
#         vertices=np.asarray(vertices),
#         faces=np.asarray(faces),
#         process=False,
#     )
#     outpath = OUTDIR / f'part_{index:02d}.obj'
#     part_mesh.export(outpath)
# print(f'Saved {len(parts)} convex OBJ files to {OUTDIR.resolve()}')

/ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/1ae1ba5dfb2a085247df6165146d5bbd/collision_meshes


In [ ]:
import numpy as np
import plotly.graph_objects as go


figure = go.Figure()

colors = [
    "#e6194b",
    "#3cb44b",
    "#4363d8",
    "#f58231",
    "#911eb4",
    "#46f0f0",
    "#f032e6",
    "#bcf60c",
    "#fabebe",
    "#008080",
    "#e6beff",
    "#9a6324",
]


for index, (vertices, faces) in enumerate(parts):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces)

    figure.add_trace(
        go.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color=colors[index % len(colors)],
            opacity=0.75,
            flatshading=True,
            name=f"part_{index:03d}",
            showlegend=True,
        )
    )


figure.update_layout(
    title=f"CoACD decomposition: {len(parts)} convex components",
    width=1000,
    height=800,
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
)

figure.show()

In [ ]:
import numpy as np
import trimesh
from pathlib import Path

output_file = Path("outputs/coacd_components.ply")
output_file.parent.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
colored_parts = []

for vertices, faces in parts:
    part = trimesh.Trimesh(
        vertices=np.asarray(vertices),
        faces=np.asarray(faces),
        process=False,
    )

    color = np.append(
        rng.integers(50, 256, size=3, dtype=np.uint8),
        np.uint8(255),
    )
    part.visual.face_colors = np.tile(color, (len(part.faces), 1))
    colored_parts.append(part)

# Merge the convex parts into one mesh without welding their vertices.
combined = trimesh.util.concatenate(colored_parts)
combined.export(output_file)

print(f"Saved {len(colored_parts)} convex parts to: {output_file.resolve()}")

# Re-decompose failed objects

In [6]:
import csv
from pathlib import Path
from pprint import pprint

failed_file = Path("data/laso_grasp_only_affordance/laso_mujoco_failed_objects.txt")
with failed_file.open("r") as f:
    reader = csv.reader(f, delimiter="\t")
    mesh_paths = [Path(row[0]) for row in reader if row]
# pprint(mesh_paths)
mesh_paths = [path for path in mesh_paths if path.is_file()]
pprint(mesh_paths)


[PosixPath('/ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/34869e23f9fdee027528ae0782b54aae/visual.obj'),
 PosixPath('/ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/d46b98f63a017578ea456f4bbbc96af9/visual.obj'),
 PosixPath('/ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/scissors/728753f3-e4e2-4a82-96e4-fc1ff2858d58/visual.obj')]


In [7]:
from coacd_decompose import main
from tqdm import tqdm

for path in tqdm(mesh_paths):
    main(
        ["--ply", f"{path}"]
    )

  0%|          | 0/3 [00:00<?, ?it/s]

INFO: Decomposing /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/34869e23f9fdee027528ae0782b54aae/visual.obj vertices=2085, faces=4209, watertight=False
[2026-10-27 12:10:10.874] [CoACD] [info] threshold               0.03
[2026-10-27 12:10:10.874] [CoACD] [info] max # convex hull       64
[2026-10-27 12:10:10.874] [CoACD] [info] preprocess mode         auto
[2026-10-27 12:10:10.874] [CoACD] [info] preprocess resolution   40
[2026-10-27 12:10:10.874] [CoACD] [info] pca                     false
[2026-10-27 12:10:10.874] [CoACD] [info] mcts max depth          3
[2026-10-27 12:10:10.874] [CoACD] [info] mcts nodes              10
[2026-10-27 12:10:10.874] [CoACD] [info] mcts iterations         30
[2026-10-27 12:10:10.874] [CoACD] [info] merge                   true
[2026-10-27 12:10:10.874] [CoACD] [info] decimate                false
[2026-10-27 12:10:10.874] [CoACD] [info] max_ch_vertex           256
[2026-10-27 12:10:10.874] [CoACD] [info] extrude               

 33%|███▎      | 1/3 [00:35<01:10, 35.29s/it]

[2026-10-27 12:10:46.071] [CoACD] [warning] Max concavity 0.0928315900437373 exceeds the threshold 0.03 due to 64 convex hull limitation
[2026-10-27 12:10:46.071] [CoACD] [info] Compute Time: 35.170258610975s
[2026-10-27 12:10:46.071] [CoACD] [info] # Convex Hulls: 64
Skipping degenerate component 15
Saved 64 convex OBJ files to /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/34869e23f9fdee027528ae0782b54aae/collision_meshes
INFO: Decomposing /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/d46b98f63a017578ea456f4bbbc96af9/visual.obj vertices=2091, faces=4235, watertight=False
[2026-10-27 12:10:46.161] [CoACD] [info] threshold               0.03
[2026-10-27 12:10:46.161] [CoACD] [info] max # convex hull       64
[2026-10-27 12:10:46.161] [CoACD] [info] preprocess mode         auto
[2026-10-27 12:10:46.161] [CoACD] [info] preprocess resolution   40
[2026-10-27 12:10:46.161] [CoACD] [info] pca                     false
[2026-10-27 12:10:46.161

 67%|██████▋   | 2/3 [01:02<00:30, 30.67s/it]

[2026-10-27 12:11:13.495] [CoACD] [warning] Max concavity 0.08427645918410423 exceeds the threshold 0.03 due to 64 convex hull limitation
[2026-10-27 12:11:13.495] [CoACD] [info] Compute Time: 27.30220333999023s
[2026-10-27 12:11:13.495] [CoACD] [info] # Convex Hulls: 64
Saved 64 convex OBJ files to /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/mug/d46b98f63a017578ea456f4bbbc96af9/collision_meshes
INFO: Decomposing /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/scissors/728753f3-e4e2-4a82-96e4-fc1ff2858d58/visual.obj vertices=2049, faces=4179, watertight=False
[2026-10-27 12:11:13.596] [CoACD] [info] threshold               0.03
[2026-10-27 12:11:13.596] [CoACD] [info] max # convex hull       64
[2026-10-27 12:11:13.596] [CoACD] [info] preprocess mode         auto
[2026-10-27 12:11:13.596] [CoACD] [info] preprocess resolution   40
[2026-10-27 12:11:13.596] [CoACD] [info] pca                     false
[2026-10-27 12:11:13.596] [CoACD] [info] mcts

100%|██████████| 3/3 [01:04<00:00, 21.39s/it]

[2026-10-27 12:11:15.012] [CoACD] [info] Compute Time: 1.3992101390031166s
[2026-10-27 12:11:15.012] [CoACD] [info] # Convex Hulls: 20
Saved 20 convex OBJ files to /ros2_ws/not-ros-pkg/easy_handover/data/laso_grasp_only_affordance/scissors/728753f3-e4e2-4a82-96e4-fc1ff2858d58/collision_meshes
